# Snowflake Connection

Connect to Snowflake with credentials read from a `.env` file — nothing hardcoded.

`.env` -> connection parameters -> session -> verify context.

---
## 1. Check the environment

Confirms the kernel already has the packages. Only install if something says `MISSING`.

In [ ]:
import sys

print("Kernel:", sys.executable)

for package in ("snowflake.snowpark", "dotenv", "pandas"):
    try:
        __import__(package)
        print(f"OK      {package}")
    except ImportError:
        print(f"MISSING {package}")

# Run this line only if something above is MISSING:
%pip install snowflake-snowpark-python python-dotenv pandas

---
## 2. Imports

In [ ]:
import os
import time
from pathlib import Path

from dotenv import load_dotenv
from snowflake.snowpark import Session

print("Imports OK")

---
## 3. The `.env` file

Keep `.env` at the repo root. Plain `KEY=VALUE`, no quotes, no spaces around `=`:

```dotenv
SNOWFLAKE_ACCOUNT=<ORG_NAME>-<ACCOUNT_NAME>
SNOWFLAKE_USER=<username>
SNOWFLAKE_PASSWORD=<password>
SNOWFLAKE_ROLE=<role>
SNOWFLAKE_WH=<warehouse>
SNOWFLAKE_DB=<database>
SNOWFLAKE_SCHEMA=<schema>
```

Never commit `.env` — keep it in `.gitignore`.

---
## 4. Locate `.env`

`__file__` does not exist in a notebook kernel, so this walks up from the current
folder until it finds the file.

In [ ]:
try:
    NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path.cwd().resolve()


def find_dotenv_path(env: str = 'dev', start: Path = NOTEBOOK_DIR) -> Path:
    """Walk up from `start` and return the first .env.<env> or .env found."""
    candidates = [f'.env.{env}', '.env']

    for directory in [start, *start.parents]:
        for name in candidates:
            candidate = directory / name
            if candidate.is_file():
                return candidate

    raise FileNotFoundError(
        f"No {' or '.join(candidates)} found in {start} or any parent folder."
    )


print("Notebook folder :", NOTEBOOK_DIR)
print("Found .env at   :", find_dotenv_path('dev'))

---
## 5. Load the environment variables

Reads `.env` into `os.environ` so `os.getenv(...)` can see the values.
`override=True` so edits to `.env` are picked up when you re-run this cell.

In [ ]:
def set_env_variables(env: str = 'dev') -> bool:
    """Load the .env file into os.environ. Returns True if anything was loaded."""
    dotenv_path = find_dotenv_path(env)
    loaded = load_dotenv(dotenv_path=dotenv_path, override=True)

    if loaded:
        print(f"Loaded environment variables from: {dotenv_path}")
    else:
        print(f"WARNING: nothing loaded from {dotenv_path} - is it empty, or not KEY=VALUE?")

    return loaded


set_env_variables('dev')

---
## 6. Configuration parameters

| Parameter | Env variable |
|-----------|--------------|
| `account` | `SNOWFLAKE_ACCOUNT` |
| `user` | `SNOWFLAKE_USER` |
| `password` | `SNOWFLAKE_PASSWORD` |
| `role` | `SNOWFLAKE_ROLE` |
| `warehouse` | `SNOWFLAKE_WH` |
| `database` | `SNOWFLAKE_DB` |
| `schema` | `SNOWFLAKE_SCHEMA` |

The password is masked before printing, so it never lands in saved cell output.

In [ ]:
def get_connection_parameters() -> dict:
    """Read the Snowflake settings from the environment."""
    return {
        "account":   os.getenv('SNOWFLAKE_ACCOUNT'),
        "user":      os.getenv('SNOWFLAKE_USER'),
        "password":  os.getenv('SNOWFLAKE_PASSWORD'),
        "role":      os.getenv('SNOWFLAKE_ROLE'),
        "warehouse": os.getenv('SNOWFLAKE_WH'),
        "database":  os.getenv('SNOWFLAKE_DB'),
        "schema":    os.getenv('SNOWFLAKE_SCHEMA'),
        # "authenticator": os.getenv('SNOWFLAKE_AUTHENTICATOR'),
    }


SECRET_KEYS = {"password", "private_key", "token", "passcode"}


def mask_secrets(parameters: dict) -> dict:
    """Return a copy of `parameters` that is safe to print."""
    return {
        key: (f"{'*' * 8} (len={len(value)})" if key in SECRET_KEYS and value else value)
        for key, value in parameters.items()
    }


def validate_connection_parameters(parameters: dict) -> None:
    """Raise a clear error if a required parameter is empty."""
    required = ["account", "user", "role", "warehouse", "database", "schema"]
    missing = [key for key in required if not parameters.get(key)]

    if not parameters.get("password") and not parameters.get("authenticator"):
        missing.append("password (or authenticator)")

    if missing:
        raise ValueError(
            "Missing connection parameter(s): " + ", ".join(missing) +
            " - check the matching SNOWFLAKE_* keys in your .env file."
        )


params = get_connection_parameters()

for key, value in mask_secrets(params).items():
    print(f"  {key:<10} : {value}")

validate_connection_parameters(params)
print("\nAll required parameters present.")

---
## 7. Create the session

`Session` is the handle used for every query afterwards.

Error `250001` is usually a transient network hiccup, so the connect is retried
a few times before giving up.

In [ ]:
def get_session(env: str = 'dev', attempts: int = 3, delay: int = 5) -> Session:
    """
    PURPOSE:
        Gets login information from the .env file and connects to
        the server. Returns the session object.
    RETURNS:
        A session.
    """
    set_env_variables(env)

    print("try init connection snowflake")
    connection_parameters = get_connection_parameters()
    validate_connection_parameters(connection_parameters)
    print(mask_secrets(connection_parameters))

    for attempt in range(1, attempts + 1):
        try:
            session = Session.builder.configs(connection_parameters).create()
            print("Connection established.")
            return session
        except Exception as exc:
            if attempt == attempts:
                raise
            print(f"Attempt {attempt}/{attempts} failed ({type(exc).__name__}) - retrying in {delay}s")
            time.sleep(delay)

In [ ]:
session = get_session('dev')
session

---
## 8. Verify the connection and context

Confirms which account, role, warehouse, database and schema the session landed on.

Note: no trailing `;` inside `session.sql(...)` — Snowpark wraps the query in a
subquery and a semicolon breaks it.

In [ ]:
def get_current_account_url(session: Session) -> str:
    """Return the account URL of the session."""
    account_url_query = """
        SELECT
        CURRENT_ORGANIZATION_NAME() || '-' || CURRENT_ACCOUNT_NAME() || '.snowflakecomputing.com'
        AS ACCOUNT_URL
    """
    account_url = session.sql(account_url_query).collect()
    return account_url[0]['ACCOUNT_URL']


def get_current_database(session: Session) -> str:
    """Return the current database of the session."""
    current_db_query = "SELECT CURRENT_DATABASE() AS CURRENT_DB"
    current_db = session.sql(current_db_query).collect()
    return current_db[0]['CURRENT_DB']


print("Account URL      :", get_current_account_url(session))
print("Current database :", get_current_database(session))

Full session context in one query:

In [ ]:
context_query = """
    SELECT
        CURRENT_ACCOUNT()   AS ACCOUNT,
        CURRENT_USER()      AS USER,
        CURRENT_ROLE()      AS ROLE,
        CURRENT_WAREHOUSE() AS WAREHOUSE,
        CURRENT_DATABASE()  AS DATABASE,
        CURRENT_SCHEMA()    AS SCHEMA,
        CURRENT_REGION()    AS REGION,
        CURRENT_VERSION()   AS VERSION
"""

context_df = session.sql(context_query).to_pandas()
context_df.T.rename(columns={0: 'VALUE'})

Does the session match what `.env` asked for? `MISMATCH` usually means the role
has no access to the requested object.

In [ ]:
context = context_df.iloc[0]

checks = [
    ("ROLE",      os.getenv('SNOWFLAKE_ROLE'),   context['ROLE']),
    ("WAREHOUSE", os.getenv('SNOWFLAKE_WH'),     context['WAREHOUSE']),
    ("DATABASE",  os.getenv('SNOWFLAKE_DB'),     context['DATABASE']),
    ("SCHEMA",    os.getenv('SNOWFLAKE_SCHEMA'), context['SCHEMA']),
]

print(f"{'PARAMETER':<12}{'REQUESTED (.env)':<32}{'ACTUAL (session)':<32}STATUS")
print("-" * 88)
for name, requested, actual in checks:
    status = "OK" if (requested or '').upper() == (actual or '').upper() else "MISMATCH"
    print(f"{name:<12}{str(requested):<32}{str(actual):<32}{status}")

A live query, to confirm the warehouse is running:

In [ ]:
session.sql("SELECT CURRENT_TIMESTAMP() AS CONNECTED_AT").show()

---
## 9. Close the session

In [ ]:
session.close()
print("Session closed.")

---
## How to run this notebook

1. **Select the kernel** — top-right of the notebook, *Select Kernel* ->
   *Python Environments* -> the Anaconda `base` interpreter
   (`C:\Users\BhavanaVemula\anaconda3\python.exe`). Cell 1 prints the kernel path;
   it must match, or the packages will show `MISSING`.
2. **Check `.env`** exists at the repo root in `KEY=VALUE` form (section 3).
3. **Run all cells in order**, top to bottom (*Run All*). Sections 4-7 define the
   functions the later cells use, so skipping one gives a `NameError`.
4. **Expected output**: section 6 prints all seven parameters with the password
   masked, section 7 prints `Connection established.`, section 8 shows the context
   table with every row `OK`.
5. **Before committing**: *Clear All Outputs*, so the account details do not get
   saved into the file.

## Troubleshooting

| Error | Fix |
|-------|-----|
| `MISSING snowflake.snowpark` | Wrong kernel selected — redo step 1 |
| `No .env found` | Put `.env` at the repo root |
| `Missing connection parameter(s)` | A `SNOWFLAKE_*` key is absent from `.env` |
| `250001 Could not connect` | Check `SNOWFLAKE_ACCOUNT` is `<ORG>-<ACCOUNT>`, not a URL; otherwise transient, just re-run |
| `Incorrect username or password` | Fix the value in `.env`, re-run section 5 |
| `Role ... does not exist or not authorized` | Ask an admin to grant the role |
| `unexpected ';'` | Remove the trailing `;` from the SQL string |
| `NameError: session is not defined` | Section 7 did not run — run the cells in order |
| `.env` edits have no effect | Re-run section 5, or restart the kernel |